# LeRobot 数据格式解析

> 目标：用本地样例把 **LeRobot 目录结构 / schema / 一帧样本 / 视频索引** 走通一遍。  
> 样例路径相对于本仓库 `robot-dataset-demos/`。

**依赖**：`pandas` `pyarrow`（见 `requirements.txt`）。可选：`lerobot` 官方库、`matplotlib`。

**推荐样例**（二选一）：
1. `samples/lerobot_minimal` — 离线最小结构（无 MP4）
2. `samples/lerobot_aloha_sim_hf` — HF 真实 ALOHA sim（有视频，约 50–70MB）

```bash
# 若还没有 HF 样例
./scripts/download_samples.sh --lerobot-only
# 国内可: export HF_ENDPOINT=https://hf-mirror.com
```


## 1. 一句话：LeRobot 是什么

LeRobot = Hugging Face 机器人学习生态的 **数据集目录约定**：

| 部件 | 存什么 |
|------|--------|
| `meta/info.json` | schema：字段名、shape、fps、路径模板（≈ COCO 的 categories+info） |
| `data/**/*.parquet` | **每一行 = 一帧**：state / action / index… |
| `videos/**/*.mp4` | 图像不直接塞进 parquet，按相机 key 存视频 |

训练时 loader 用 `index` / `frame_index` 把 parquet 行和视频帧对齐。


In [ ]:
from pathlib import Path
import json

# notebook 在 notebooks/ 下；样例在 ../samples
REPO = Path("..").resolve()
SAMPLES = REPO / "samples"

CANDIDATES = [
    SAMPLES / "lerobot_aloha_sim_hf",
    SAMPLES / "lerobot_minimal",
]
ROOT = next((p for p in CANDIDATES if (p / "meta" / "info.json").exists()), None)
assert ROOT is not None, f"找不到 LeRobot 样例，请先 download 或 create_minimal。候选: {CANDIDATES}"
print("Using:", ROOT)
print("Children:", sorted(p.name for p in ROOT.iterdir() if not p.name.startswith(".")))


## 2. 读 `meta/info.json`（先搞懂 schema）

`info.json` 告诉你：有哪些 feature、图像是 video 还是 image、action 几维、关节名是什么。


In [ ]:
info = json.loads((ROOT / "meta" / "info.json").read_text(encoding="utf-8"))

summary_keys = [
    "codebase_version", "robot_type", "total_episodes", "total_frames",
    "total_tasks", "fps", "chunks_size", "data_path", "video_path",
]
print("=== Dataset summary ===")
for k in summary_keys:
    if k in info:
        print(f"  {k}: {info[k]}")

print("\n=== Features ===")
for name, feat in info.get("features", {}).items():
    shape = feat.get("shape")
    dtype = feat.get("dtype")
    names = feat.get("names")
    print(f"  {name}: dtype={dtype}, shape={shape}")
    if isinstance(names, dict) and "motors" in names:
        print(f"      motors({len(names['motors'])}): {names['motors'][:4]} ...")
    elif names:
        print(f"      names: {names}")


### 目录树直觉

```text
my_dataset/
├── meta/
│   ├── info.json          # schema
│   └── tasks.parquet      # 可选：task 文本 ↔ task_index
├── data/
│   └── chunk-000/
│       └── file-000.parquet   # v3: 多 episode 可同文件
│       # 或 episode_000000.parquet  # v2: 一 episode 一文件
└── videos/
    └── observation.images.top/
        └── chunk-000/
            └── file-000.mp4
```

**v2 vs v3**：字段语义相同；v3 更常「多个 episode 挤在一个 parquet / mp4 file」里，靠 `episode_index` 区分。


In [ ]:
# 列出 data / videos 实际文件
data_files = sorted((ROOT / "data").rglob("*.parquet")) if (ROOT / "data").exists() else []
video_files = sorted((ROOT / "videos").rglob("*.mp4")) if (ROOT / "videos").exists() else []
print(f"parquet files: {len(data_files)}")
for f in data_files[:5]:
    print(" ", f.relative_to(ROOT))
print(f"mp4 files: {len(video_files)}")
for f in video_files[:5]:
    print(" ", f.relative_to(ROOT))

tasks_path = ROOT / "meta" / "tasks.parquet"
if tasks_path.exists():
    import pandas as pd
    tasks = pd.read_parquet(tasks_path)
    print("\ntasks.parquet:")
    print(tasks.head())


## 3. 解析 Parquet：一行 = 一帧

重点字段（ALOHA sim 样例）：
- `observation.state`：本体感觉（本样例 14 维关节）
- `action`：该步动作目标（本样例也是 14 维）
- `episode_index` / `frame_index` / `timestamp` / `index`
- `task_index`：指向任务文本
- `next.done`：是否 episode 结束


In [ ]:
import pandas as pd
import numpy as np

pq = data_files[0]
df = pd.read_parquet(pq)
print("file:", pq.relative_to(ROOT))
print("rows x cols:", df.shape)
print("columns:", list(df.columns))

# 若含多 episode，先看分布
if "episode_index" in df.columns:
    print("\nepisode_index counts (head):")
    print(df["episode_index"].value_counts().sort_index().head())

ep0 = df[df["episode_index"] == 0].copy() if "episode_index" in df.columns else df
print(f"\nEpisode 0 frames: {len(ep0)}")
print("frame_index range:", ep0["frame_index"].min(), "→", ep0["frame_index"].max())
print("timestamp range:", float(ep0["timestamp"].min()), "→", float(ep0["timestamp"].max()), "sec")


In [ ]:
# 看第 0 帧的向量
row = ep0.iloc[0]
state = np.asarray(row["observation.state"], dtype=np.float32)
action = np.asarray(row["action"], dtype=np.float32)
print("observation.state shape:", state.shape, "\n ", np.round(state, 4))
print("action shape:", action.shape, "\n ", np.round(action, 4))
print("task_index:", int(row["task_index"]) if "task_index" in row else None)
print("next.done:", bool(row["next.done"]) if "next.done" in row else None)

# 与 info.json 里 motors 名字对齐（若有）
motors = (
    info.get("features", {})
    .get("action", {})
    .get("names", {})
)
if isinstance(motors, dict):
    motors = motors.get("motors")
if motors and len(motors) == len(action):
    print("\naction by joint name:")
    for name, val in zip(motors, action):
        print(f"  {name:24s} {val: .4f}")


## 4. 动作曲线可视化（理解「时间序列」）

对 episode 0 画几维关节 / 动作随时间变化——直觉上这就是模仿学习的监督信号 a_t。



In [ ]:
import matplotlib.pyplot as plt

actions = np.stack(ep0["action"].to_list()).astype(np.float32)  # [T, D]
states = np.stack(ep0["observation.state"].to_list()).astype(np.float32)
t = ep0["timestamp"].to_numpy() if "timestamp" in ep0.columns else np.arange(len(ep0))

# 选几个维度画
dims = list(range(min(4, actions.shape[1])))
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
for d in dims:
    label = motors[d] if motors and d < len(motors) else f"dim{d}"
    axes[0].plot(t, states[:, d], label=label, alpha=0.85)
    axes[1].plot(t, actions[:, d], label=label, alpha=0.85)
axes[0].set_ylabel("state")
axes[1].set_ylabel("action")
axes[1].set_xlabel("time (s)" if "timestamp" in ep0.columns else "frame")
axes[0].legend(ncol=4, fontsize=8)
axes[1].legend(ncol=4, fontsize=8)
axes[0].set_title(f"Episode 0 · state / action (first {len(dims)} dims)")
plt.tight_layout()
plt.show()

print("action mean:", np.round(actions.mean(0)[:6], 4), "...")
print("action std :", np.round(actions.std(0)[:6], 4), "...")


## 5. 视频怎么和 parquet 对齐

图像一般在 `videos/<camera_key>/...mp4`。  
**parquet 里通常不存像素**，只存索引；训练时按 `frame_index` 取第 N 帧。

下面用 OpenCV 读一帧（若已装 `opencv-python`）；没有就跳过。


In [ ]:
cam_keys = [
    k for k, v in info.get("features", {}).items()
    if v.get("dtype") == "video" or "images" in k
]
print("camera / video feature keys:", cam_keys)

frame_img = None
if video_files and cam_keys:
    try:
        import cv2
        vpath = video_files[0]
        cap = cv2.VideoCapture(str(vpath))
        ok, frame_bgr = cap.read()
        cap.release()
        if ok:
            frame_img = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            print("read", vpath.relative_to(ROOT), "shape", frame_img.shape)
            plt.figure(figsize=(5, 4))
            plt.imshow(frame_img)
            plt.axis("off")
            plt.title("first frame of " + vpath.name)
            plt.show()
        else:
            print("Failed to read frame from", vpath)
    except ImportError:
        print("opencv 未安装：pip install opencv-python-headless 后再跑本格")
else:
    print("无视频文件（minimal 样例正常）。换 lerobot_aloha_sim_hf 可看图。")


## 6.（可选）用官方 `lerobot` 库加载

若已 `pip install lerobot`，可用 Dataset 类按 index 取样本（会自动解码视频）。  
接口随版本可能略有差异。


In [ ]:
try:
    from lerobot.common.datasets.lerobot_dataset import LeRobotDataset
    # 本地目录: 有的版本支持 root= ; 有的要 repo_id
    ds = LeRobotDataset(str(ROOT))
    sample = ds[0]
    print("sample keys:", list(sample.keys()) if isinstance(sample, dict) else type(sample))
    for k, v in (sample.items() if isinstance(sample, dict) else []):
        if hasattr(v, "shape"):
            print(f"  {k}: shape={tuple(v.shape)} dtype={getattr(v, 'dtype', None)}")
        else:
            print(f"  {k}: {type(v).__name__}")
except Exception as e:
    print("官方 lerobot 加载跳过 / 失败（不影响前面手工解析）:")
    print(" ", type(e).__name__, e)


## 7. 对照清单：你应该能回答

1. **一行 parquet 对应什么？** → 一个时间步（一帧）。
2. **图像在哪？** → `videos/` 里的 mp4，不在 action 列里。
3. **本 ALOHA 样例 action 是 delta EEF 吗？** → **不是**，是 **14 维关节目标**（见 motors 名）。
4. **语言指令在哪？** → 常在 `meta/tasks.parquet`，帧上只有 `task_index`。
5. **和 RLDS 最大差别？** → LeRobot 偏 PyTorch/HF、parquet+mp4；RLDS 偏 TF/JAX、Episode 嵌套 steps 的 TFRecord。

下一步：打开同目录 `02_rlds_format.ipynb`。
